In [49]:
import numpy as np

print("Ambiente configurado com sucesso!")
print("Versão do NumPy:", np.__version__)

Ambiente configurado com sucesso!
Versão do NumPy: 2.5.3


# AT1 — Perceptron e KNN em Prática

## Desafio 1 — Classificação de Fraudes com Perceptron

Treinar um Perceptron para classificar transações financeiras como legítimas (0) ou suspeitas (1).

In [50]:
# Dados históricos: [valor normalizado, frequência de operações]
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

# Classes: 0 = legítima, 1 = suspeita
y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])

print("Transações de treinamento:", len(X_treino_fraude))
print("Classes:", y_treino_fraude)

Transações de treinamento: 8
Classes: [0 0 0 0 1 1 1 1]


In [51]:
def treinar_perceptron(X, y, taxa_aprendizado=0.1, epocas=100):
    # Inicializa os pesos e o viés
    pesos = np.ones(X.shape[1])
    vies = 1.0

    for epoca in range(epocas):
        erros = 0

        for amostra, classe_real in zip(X, y):
            # Calcula a saída do Perceptron
            resultado = np.dot(amostra, pesos) + vies
            previsao = 1 if resultado >= 0 else 0

            # Calcula o erro e ajusta os parâmetros
            erro = classe_real - previsao

            if erro != 0:
                pesos += taxa_aprendizado * erro * amostra
                vies += taxa_aprendizado * erro
                erros += 1

        # Interrompe quando não existem mais erros
        if erros == 0:
            print(f"Treinamento concluído na época {epoca + 1}")
            break

    return pesos, vies

In [52]:
def prever_perceptron(amostra, pesos, vies):
    resultado = np.dot(amostra, pesos) + vies
    return 1 if resultado >= 0 else 0

In [53]:
# Treinamento
pesos, vies = treinar_perceptron(
    X_treino_fraude,
    y_treino_fraude,
    taxa_aprendizado=0.1,
    epocas=100
)

print("Pesos finais:", pesos)
print("Viés final:", vies)

# Novas transações
transacao_A = np.array([5.0, 4.0])
transacao_B = np.array([7.0, 6.5])

# Classificação
for nome, transacao in [
    ("A", transacao_A),
    ("B", transacao_B)
]:
    classe = prever_perceptron(transacao, pesos, vies)
    diagnostico = "Suspeita" if classe == 1 else "Legítima"

    print(f"Transação {nome}: {diagnostico}")

Treinamento concluído na época 12
Pesos finais: [-5.55111512e-16  4.00000000e-01]
Viés final: -1.2
Transação A: Suspeita
Transação B: Suspeita


In [54]:
previsoes = np.array([
    prever_perceptron(amostra, pesos, vies)
    for amostra in X_treino_fraude
])

acertos = np.sum(previsoes == y_treino_fraude)

print(f"Acertos: {acertos}/{len(y_treino_fraude)}")
print(f"Acurácia: {acertos / len(y_treino_fraude):.0%}")

Acertos: 8/8
Acurácia: 100%


# Desafio 2 — Predição de Risco de Churn com KNN

Implementar um classificador KNN para prever o risco de cancelamento de clientes de uma empresa SaaS.

O algoritmo utiliza as distâncias "Euclidiana" e "Manhattan" para encontrar os K vizinhos mais próximos e determinar
a classe por votação majoritária.

In [55]:
# Dados históricos: [dias de inatividade, chamados críticos]
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

# Classes: 0 = Baixo Risco, 1 = Alto Risco
y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

print("Clientes históricos:", len(X_treino_churn))

Clientes históricos: 8


In [56]:
def classificar_knn(X, y, ponto, k, metrica="euclidiana"):
    # Calcula as distâncias de forma vetorizada
    diferencas = X - ponto

    if metrica == "euclidiana":
        distancias = np.sqrt(np.sum(diferencas ** 2, axis=1))

    elif metrica == "manhattan":
        distancias = np.sum(np.abs(diferencas), axis=1)

    else:
        raise ValueError("Métrica inválida.")

    # Seleciona os K vizinhos mais próximos
    indices = np.argsort(distancias)[:k]

    # Obtém as classes dos vizinhos
    classes_vizinhas = y[indices]

    # Determina a classe por maioria de votos
    votos = np.bincount(classes_vizinhas)
    classe_predita = np.argmax(votos)

    return classe_predita, indices, distancias[indices]

In [57]:
# Clientes que serão avaliados
cliente_1 = np.array([13.0, 10.0])
cliente_2 = np.array([7.0, 1.5])

# Configuração dos testes
k = 3

testes = [
    ("Cliente 1", cliente_1, "euclidiana"),
    ("Cliente 2", cliente_2, "manhattan")
]

for nome, cliente, metrica in testes:
    classe, indices, distancias = classificar_knn(
        X_treino_churn,
        y_treino_churn,
        cliente,
        k,
        metrica
    )

    diagnostico = "Alto Risco" if classe == 1 else "Baixo Risco"

    print(f"\n{nome}")
    print(f"Métrica: {metrica}")
    print(f"Classe predita: {diagnostico}")
    print(f"Índices dos vizinhos: {indices}")
    print(f"Distâncias: {np.round(distancias, 2)}")


Cliente 1
Métrica: euclidiana
Classe predita: Alto Risco
Índices dos vizinhos: [4 5 6]
Distâncias: [6.32 7.07 9.22]

Cliente 2
Métrica: manhattan
Classe predita: Baixo Risco
Índices dos vizinhos: [3 2 1]
Distâncias: [1.5 2.5 5.5]


## Desafio 3 — Recomendação de Servidores Cloud

Para esse desafio o objetivo é recomendar os servidores mais similares às necessidades de hardware de uma aplicação.

A recomendação utiliza distância "Euclidiana" para comparar vCPUs, memória RAM e armazenamento SSD, selecionando os dois servidores mais próximos do perfil desejado.

In [58]:
# Catálogo: [vCPUs, memória RAM (GB), armazenamento SSD (GB)]
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])

# Nomes comerciais dos servidores
nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training"
]

print("Servidores disponíveis:", len(catalogo_servidores))

Servidores disponíveis: 6


In [59]:
def recomendar_servidores(catalogo, perfil, k=2):
    # Calcula a diferença entre o perfil e todos os servidores
    diferencas = catalogo - perfil

    # Calcula as distâncias Euclidianas de forma vetorizada
    distancias = np.sqrt(np.sum(diferencas ** 2, axis=1))

    # Ordena e seleciona os K servidores mais próximos
    indices = np.argsort(distancias)[:k]

    return indices, distancias[indices]

In [60]:
# Perfil desejado: [vCPUs, RAM (GB), SSD (GB)]
perfil_desejado = np.array([12.0, 28.0, 450.0])

# Quantidade de recomendações
k = 2

# Executa o recomendador
indices, distancias = recomendar_servidores(
    catalogo_servidores,
    perfil_desejado,
    k
)

# Exibe o ranking
print("=== RANKING DE SERVIDORES RECOMENDADOS ===")

for posicao, (indice, distancia) in enumerate(
    zip(indices, distancias), start=1
):
    nome = nomes_servidores[indice]
    cpu, ram, ssd = catalogo_servidores[indice]

    print(f"\n{posicao}º lugar — {nome}")
    print(f"vCPUs: {cpu:.0f}")
    print(f"RAM: {ram:.0f} GB")
    print(f"SSD: {ssd:.0f} GB")
    print(f"Distância Euclidiana: {distancia:.2f}")

=== RANKING DE SERVIDORES RECOMENDADOS ===

1º lugar — Database Enterprise
vCPUs: 16
RAM: 32 GB
SSD: 500 GB
Distância Euclidiana: 50.32

2º lugar — Medium Backend & Cache
vCPUs: 8
RAM: 16 GB
SSD: 250 GB
Distância Euclidiana: 200.40
